# **Course**: Deep Learning

[<img align="right" width="400" height="100" src="https://www.tu-braunschweig.de/typo3conf/ext/tu_braunschweig/Resources/Public/Images/Logos/tu_braunschweig_logo.svg">](https://www.tu-braunschweig.de/en/)

[Mehdi Maboudi](https://www.tu-braunschweig.de/en/igp/staff/mehdi-maboudi) \([m.maboudi@tu-bs.de](m.maboudi@tu-bs.de)) and [Pedro Achanccaray](https://www.tu-braunschweig.de/en/igp/staff/pedro-diaz) (p.diaz@tu-bs.de)

[Technical University of Braunschweig](https://www.tu-braunschweig.de/en/)  
[Institute of Geodesy and Photogrammetry](https://www.tu-braunschweig.de/igp)

# **Assignment 01:** **E**xploratory **D**ata **A**nalysis on GTSR dataset(**EDA**)

## Dataset Download and Setup

The dataset was downloaded directly within the Colab environment using a public archive link. The compressed file containing the training images was extracted into a designated data directory for further processing.

To ensure a clean and organized workflow:
- A dedicated `/content/data/` directory was created
- The dataset archive was downloaded and extracted into this directory
- Temporary files were removed after extraction to free up storage

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Move to the default writable Colab directory
%cd /content/
!mkdir -p /content/data

# Download the images
!wget https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip
!unzip -q GTSRB_Final_Training_Images.zip -d /content/data/

!rm GTSRB_Final_Training_Images.zip

/content
--2026-04-22 23:13:25--  https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip
Resolving sid.erda.dk (sid.erda.dk)... 130.225.104.13
Connecting to sid.erda.dk (sid.erda.dk)|130.225.104.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 276294756 (263M) [application/zip]
Saving to: ‘GTSRB_Final_Training_Images.zip’

GTSRB_Final_Trainin 100%[===================>] 263.50M  14.9MB/s    in 21s     

2026-04-22 23:13:47 (12.6 MB/s) - ‘GTSRB_Final_Training_Images.zip’ saved [276294756/276294756]

replace /content/data/GTSRB/Final_Training/Images/00000/00000_00000.ppm? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
from glob import glob
from natsort import natsorted
from PIL import Image

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

## Dataset Overview

The dataset consists of images of traffic signs categorized into different classes. Each class represents a specific traffic sign type. The images are organized into class-wise directories, where each folder corresponds to one class label.

To prepare the dataset for analysis, all image paths were collected recursively and stored in a structured dataframe. The class labels were extracted from the folder names and mapped to integer values to ensure compatibility with machine learning models.

The dataset contains:
- **43 classes**
- **39,209 images**

This confirms that the problem is a **multi-class image classification task**.


In [ ]:
# Correct path in Colab
DATASET_PATH = "/content/data/GTSRB/Final_Training/Images"

# Collect all image paths
list_img = glob(os.path.join(DATASET_PATH, "**", "*.ppm"), recursive=True)

# Natural sorting
list_img = natsorted(list_img, key=lambda x: x.lower())
df = pd.DataFrame(list_img, columns=["path_image"])

# Extract class label
df["class_str"] = df["path_image"].apply(lambda x: os.path.basename(os.path.dirname(x)))

# Map to integers
classes = np.unique(df["class_str"].values)
classes_dict = dict(zip(classes, range(len(classes))))

df["class_int"] = df["class_str"].apply(lambda x: classes_dict[x])
# Shuffle
df = df.sample(frac=1).reset_index(drop=True)

print("Number of images:", len(df))
print("Number of classes:", len(classes))

df.head()

## Class Distribution

The distribution of images across classes was analyzed to check for imbalance. The number of samples per class varies significantly:

- Minimum samples in a class: **210**
- Maximum samples in a class: **2250**

This indicates a **long-tail distribution**, where some classes have significantly more samples than others. Such imbalance can bias the model toward frequently occurring classes and lead to poor performance on underrepresented classes.

In [ ]:
# Count images per class
class_counts = df["class_int"].value_counts().sort_index()

print(class_counts.describe())
print("Max images in a class:", class_counts.max())
print("Min images in a class:", class_counts.min())

In [ ]:
plt.figure()
plt.bar(class_counts.index, class_counts.values)
plt.xlabel("Class ID")
plt.ylabel("Number of Images")
plt.title("Class Distribution")
plt.show()

## Image Size Analysis

The dimensions of all images were analyzed to understand variability in input size. The results show:

- Minimum size: **25 × 25**
- Maximum size: **243 × 225**
- Average size: approximately **51 × 50**

This wide variation in image dimensions indicates that the dataset does not have a uniform resolution. Since convolutional neural networks require fixed-size inputs, **resizing is a necessary preprocessing step** before training.

In [ ]:
widths = []
heights = []

for img_path in df["path_image"]:
    img = Image.open(img_path)
    w, h = img.size
    widths.append(w)
    heights.append(h)

widths = np.array(widths)
heights = np.array(heights)

print("Min width:", widths.min())
print("Max width:", widths.max())
print("Min height:", heights.min())
print("Max height:", heights.max())
print("Average width:", widths.mean())
print("Average height:", heights.mean())

## Visual Inspection

A subset of images from different classes was visualized to qualitatively assess the dataset. The inspection revealed:

- Variations in lighting conditions (e.g., bright sunlight, shadows)
- Presence of motion blur in some images
- Rotations and perspective distortions
- Partial occlusions of traffic signs
- Background clutter such as trees, roads, and vehicles

These observations indicate that the dataset reflects **real-world driving conditions**, making the classification task more challenging and requiring robust models.

In [ ]:
classes_sample = np.random.choice(df["class_int"].unique(), 5, replace=False)

plt.figure(figsize=(12, 8))

for i, c in enumerate(classes_sample):
    sample_imgs = df[df["class_int"] == c]["path_image"].sample(5).values
    
    for j, img_path in enumerate(sample_imgs):
        img = Image.open(img_path)
        
        plt.subplot(5, 5, i*5 + j + 1)
        plt.imshow(img)
        plt.axis("off")

plt.tight_layout()
plt.show()

## Class Labels and Semantic Meaning

Each class corresponds to a specific traffic sign category, and labels are derived directly from the folder structure. The mapping between class identifiers and integer labels is consistent across the dataset.

Since many traffic signs share similar visual characteristics (e.g., speed limit signs with different numbers, warning signs with similar shapes), the classification problem involves **fine-grained distinctions between visually similar classes**.

In [ ]:
print(df["class_str"].unique())
df.head()

In [ ]:
# pick a few random classes
sample_classes = df["class_int"].unique()[:5]

plt.figure(figsize=(12, 8))

for i, c in enumerate(sample_classes):
    samples = df[df["class_int"] == c]["path_image"].sample(3).values
    
    for j, img_path in enumerate(samples):
        img = Image.open(img_path)
        
        plt.subplot(len(sample_classes), 3, i*3 + j + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Class {c}")

plt.tight_layout()
plt.show()

## Color and Feature Analysis

The distribution of RGB pixel values was analyzed to understand the importance of color information. The results show that pixel intensities are not uniformly distributed across channels, indicating structured color patterns.

Traffic signs rely heavily on:
- **Color** (e.g., red for warnings, blue for mandatory signs)
- **Shape** (e.g., circular, triangular)

This confirms that **color is a critical feature** for classification. Converting images to grayscale would lead to loss of important discriminative information, and therefore RGB representation should be preserved.

In [ ]:
sample_pixels = []

for img_path in df["path_image"].sample(200):
    img = Image.open(img_path)
    arr = np.array(img)
    sample_pixels.append(arr.reshape(-1, 3))

sample_pixels = np.vstack(sample_pixels)

print("Pixel matrix shape:", sample_pixels.shape)

plt.figure(figsize=(10,5))

plt.hist(sample_pixels[:, 0], bins=50, color="red", alpha=0.5, label="Red")
plt.hist(sample_pixels[:, 1], bins=50, color="green", alpha=0.5, label="Green")
plt.hist(sample_pixels[:, 2], bins=50, color="blue", alpha=0.5, label="Blue")

plt.legend()
plt.title("RGB Pixel Distribution")
plt.show()

gray_sample = Image.open(df["path_image"].iloc[0]).convert("L")
plt.imshow(gray_sample, cmap="gray")
plt.title("Grayscale Example")
plt.axis("off")
plt.show()

## Summary of Key Findings

- The dataset contains **43 classes with ~39k images**
- There is **moderate class imbalance**
- Images have **high variability in size and quality**
- The dataset reflects **real-world conditions**
- Classes exhibit **high visual similarity**
- **Color information is essential** for classification

These insights guide the design of preprocessing and model training strategies in subsequent steps.